### Vergleich Standzeiten Pro Stadt 

In diesem Notebooks werden die Plots und Vergleiche der Standzeiten erstellt.

In [29]:
import pandas as pd
import plotly.graph_objects as go 
import plotly.express as px
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [30]:
# SOLL-Daten (Tourdaten mit PLZ)
soll_raw = pd.read_csv('Soll-Daten_mit_PLZ.csv')
soll_raw.shape
soll_raw.head(3)

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,PLZ,ORT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,20457.0,Hamburg,9.98632,53.52348,0 days 00:30:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,20457.0,Hamburg,9.99383,53.54751,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,20457.0,Hamburg,9.98632,53.52348,0 days 00:30:00,2026-03-02


In [31]:
# IST-Daten aus Analyse_standzeiten notebook
ist = pd.read_csv('Ist_Ort.csv')
print(f"IST-Datensatz: {ist.shape[0]} Einträge")
ist.head(5)

IST-Datensatz: 38 Einträge


,PLZ,Stadt,Avg,Median,Stopps
0,27374,Visselhövede,52.0,51.3,10
1,26180,Rastede,46.8,45.8,5
2,21683,Stade,37.7,41.5,44
3,25541,Brunsbüttel,37.1,38.8,5
4,23795,Bad Segeberg,35.2,40.8,5


In [32]:
soll_raw['STOPP_MIN'] = pd.to_timedelta(soll_raw['STOPP_DAUER']).dt.total_seconds() / 60

soll_by_plz = (soll_raw.groupby(['PLZ', 'ORT'])['STOPP_MIN'].agg(Soll_Avg='mean',Soll_Median='median',Soll_Stopps='count').reset_index())

soll_by_plz['PLZ'] = soll_by_plz['PLZ'].astype(int)
soll_by_plz.head(8)

,PLZ,ORT,Soll_Avg,Soll_Median,Soll_Stopps
0,6237,Leuna,30.0,30.0,1
1,14195,Berlin,15.0,15.0,1
2,17509,Lubmin,30.0,30.0,2
3,18055,Rostock,15.0,15.0,1
4,18059,Papendorf,15.0,15.0,2
5,19063,Schwerin,15.0,15.0,1
6,19230,Hagenow,15.0,15.0,1
7,19246,Lüttow,15.0,15.0,1


In [33]:
ist_renamed = ist.rename(columns={
    'Avg': 'Ist_Avg',
    'Median': 'Ist_Median',
    'Stopps': 'Ist_Stopps'
})

merged = pd.merge(
    soll_by_plz,
    ist_renamed,
    left_on='PLZ',
    right_on='PLZ',
    how='inner'
)

merged['Abweichung_Avg'] = merged['Ist_Avg'] - merged['Soll_Avg']
merged['Abweichung_Median'] = merged['Ist_Median'] - merged['Soll_Median']

# Startort rausfiltern
merged = merged[merged['PLZ'] != 22419].reset_index(drop=True) #22419 Hamburg wird rausgestrichen, da es der Start- und Endpunkt der Meisten Fahrten ist und nicht wo sachen abgeladen werden

print(f"Gemeinsame PLZ-Einträge (ohne Startort 22419): {len(merged)}")
merged[['PLZ', 'ORT', 'Soll_Avg', 'Ist_Avg', 'Abweichung_Avg', 'Soll_Stopps', 'Ist_Stopps']].round(1)

Gemeinsame PLZ-Einträge (ohne Startort 22419): 37


,PLZ,ORT,Soll_Avg,Ist_Avg,Abweichung_Avg,Soll_Stopps,Ist_Stopps
0,20097,Hamburg,18.3,10.5,-7.8,3,11
1,20354,Hamburg,18.8,19.3,0.6,8,6
2,20355,Hamburg,15.0,6.7,-8.3,12,9
3,20457,Hamburg,30.0,24.9,-5.1,690,517
4,21029,Hamburg,22.2,19.1,-3.1,9,7
5,21035,Hamburg,20.0,12.6,-7.4,4,5
6,21107,Hamburg,34.3,23.9,-10.4,164,124
7,21129,Hamburg,18.9,13.6,-5.3,7,6
8,21337,Lüneburg,19.0,13.6,-5.4,5,5
9,21465,Reinbek,18.1,4.8,-13.3,8,5


In [34]:
overview = (
    merged[['PLZ', 'ORT', 'Soll_Avg', 'Soll_Median', 'Soll_Stopps', 'Ist_Avg', 'Ist_Median', 'Ist_Stopps', 'Abweichung_Avg']]
    .sort_values('Ist_Avg', ascending=False).round(1).reset_index(drop=True)
)

overview.columns = ['PLZ', 'Ort', 'Soll Ø (Min)', 'Soll Median', 'Soll Stopps', 'IST Ø (Min)', 'IST Median', 'IST Stopps', 'Abweichung Ø']
overview

,PLZ,Ort,Soll Ø (Min),Soll Median,Soll Stopps,IST Ø (Min),IST Median,IST Stopps,Abweichung Ø
0,27374,Visselhoevede,30.0,30.0,14,52.0,51.3,10,22.0
1,26180,Rastede,30.0,30.0,10,46.8,45.8,5,16.8
2,21683,Stade,37.6,30.0,50,37.7,41.5,44,0.1
3,25541,Brunsbüttel,30.0,30.0,6,37.1,38.8,5,7.1
4,23795,Bad Segeberg,30.0,30.0,8,35.2,40.8,5,5.2
5,22113,Hamburg,25.0,30.0,14,34.4,25.9,9,9.4
6,22844,Norderstedt,25.4,25.0,25,30.4,31.8,23,5.0
7,24589,Nortorf,33.5,25.0,13,28.1,21.2,9,-5.4
8,20457,Hamburg,30.0,30.0,690,24.9,21.9,517,-5.1
9,21107,Hamburg,34.3,30.0,164,23.9,21.0,124,-10.4


In [35]:
#Grouped Bar Chart SOLL vs. IST je Ort
plot_df = merged.sort_values('Ist_Avg', ascending=True).copy()
plot_df['label'] = plot_df['PLZ'].astype(str) + ' – ' + plot_df['ORT']

fig = go.Figure()

fig.add_trace(go.Bar(
    y=plot_df['label'],
    x=plot_df['Soll_Avg'],
    name='SOLL Ø',
    orientation='h',
    marker_color='#00498b',
    opacity=0.85
))

fig.add_trace(go.Bar(
    y=plot_df['label'],
    x=plot_df['Ist_Avg'],
    name='IST Ø',
    orientation='h',
    marker_color='#ff0000',
    opacity=0.75
))

fig.update_layout(
    template='plotly_white',
    barmode='group',
    title='Ø Stoppdauer je Ort: SOLL vs. IST',
    xaxis_title='Ø Stoppdauer (Minuten)',
    yaxis_title='Ort (PLZ)',
    height=max(500, len(plot_df) * 28),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

In [36]:
# Fahrer schneller als geplant top 10
top10_soll_groesser = (merged[merged['Abweichung_Avg'] < 0].nsmallest(10, 'Abweichung_Avg').sort_values('Abweichung_Avg', ascending=False) .copy())
top10_soll_groesser['label'] = top10_soll_groesser['PLZ'].astype(str) + ' – ' + top10_soll_groesser['ORT']

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    y=top10_soll_groesser['label'],
    x=top10_soll_groesser['Soll_Avg'],
    name='SOLL Ø',
    orientation='h',
    marker_color='#ff0000',
    opacity=0.85
))

fig1.add_trace(go.Bar(
    y=top10_soll_groesser['label'],
    x=top10_soll_groesser['Ist_Avg'],
    name='IST Ø',
    orientation='h',
    marker_color='#00498b',
    opacity=0.75,
    text=top10_soll_groesser['Abweichung_Avg'].round(1).apply(lambda v: f'{v:.1f} Min'),
    textposition='outside',
    textfont=dict(color='#00498b', size=11)
))

fig1.update_layout(
    template='plotly_white',
    barmode='group',
    title='Top 10 Orte: SOLL > IST (Fahrer schneller als geplant)',
    xaxis_title='Ø Stoppdauer (Minuten)',
    yaxis_title='Ort (PLZ)',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig1.show()

In [37]:
# Fahrer langsamer als geplant top 10
top10_ist_groesser = (merged[merged['Abweichung_Avg'] > 0].nlargest(10, 'Abweichung_Avg').sort_values('Abweichung_Avg', ascending=True).copy())

top10_ist_groesser['label'] = top10_ist_groesser['PLZ'].astype(str) + ' – ' + top10_ist_groesser['ORT']

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=top10_ist_groesser['label'],
    x=top10_ist_groesser['Soll_Avg'],
    name='SOLL Ø',
    orientation='h',
    marker_color='#ff0000',
    opacity=0.85
))

fig2.add_trace(go.Bar(
    y=top10_ist_groesser['label'],
    x=top10_ist_groesser['Ist_Avg'],
    name='IST Ø',
    orientation='h',
    marker_color='#00498b',
    opacity=0.75,
    text=top10_ist_groesser['Abweichung_Avg'].round(1).apply(lambda v: f'+{v:.1f} Min'),
    textposition='outside',
    textfont=dict(color='#00498b', size=11)
))

fig2.update_layout(
    template='plotly_white',
    barmode='group',
    title='Top 10 Orte: IST > SOLL (Fahrer länger als geplant)',
    xaxis_title='Ø Stoppdauer (Minuten)',
    yaxis_title='Ort (PLZ)',
    height=400,
    margin=dict(r=100),  
    xaxis=dict(
        range=[0, top10_ist_groesser['Ist_Avg'].max() * 1.3]  
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig2.show()

In [38]:
#Abweichungsplot
diff_df = merged.sort_values('Abweichung_Avg').copy()
diff_df['label'] = diff_df['PLZ'].astype(str) + ' – ' + diff_df['ORT']
bar_colors = ['#ff0000' if x > 0 else '#00498b' for x in diff_df['Abweichung_Avg']]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=diff_df['label'],
    x=diff_df['Abweichung_Avg'],
    orientation='h',
    marker_color=bar_colors,
    opacity=0.85,
    text=diff_df['Abweichung_Avg'].round(1).apply(lambda v: f'{v:+.1f} Min'),
    textposition='outside'
))

fig.add_vline(x=0, line_width=1.2, line_color='black')

fig.update_layout(
    template='plotly_white',
    title='Abweichung IST – SOLL je Ort (Ø Stoppdauer)',
    xaxis_title='IST minus SOLL (Minuten)',
    yaxis_title='Ort (PLZ)',
    height=max(500, len(diff_df) * 28),
    showlegend=False
)

fig.show()

In [39]:
#Scatterplot
merged['label'] = merged['PLZ'].astype(str) + ' – ' + merged['ORT']

fig = go.Figure()

# Diagonale (SOLL = IST)
lim_max = max(merged['Soll_Avg'].max(), merged['Ist_Avg'].max()) * 1.1
fig.add_trace(go.Scatter(
    x=[0, lim_max], y=[0, lim_max],
    mode='lines',
    line=dict(color='grey', dash='dash', width=1),
    name='SOLL = IST'
))

# Datenpunkte sind städte
fig.add_trace(go.Scatter(
    x=merged['Soll_Avg'],
    y=merged['Ist_Avg'],
    mode='markers+text',
    text=merged['ORT'],
    textposition='top right',
    textfont=dict(size=9, color='#00498b'),
    marker=dict(
        size=merged['Ist_Stopps'] / merged['Ist_Stopps'].max() * 30 + 8,
        color=merged['Abweichung_Avg'],
        colorscale=[[0, '#00498b'], [0.5, '#f0f0f0'], [1, '#ff0000']],
        colorbar=dict(title='Abweichung<br>(IST−SOLL) Min'),
        showscale=True,
        line=dict(color='white', width=0.5)
    ),
    name='Orte'
))

fig.update_layout(
    template='plotly_white',
    title='SOLL vs. IST je PLZ – Punktgröße = Anzahl IST-Stopps',
    xaxis_title='SOLL Ø Stoppdauer (Min)',
    yaxis_title='IST Ø Stoppdauer (Min)',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

Nur wenige Punkte liegen über der Diagonalen, also die Städte in denen Fahrer mehr Zeit brauchten.

In [40]:
laenger = merged[merged['Abweichung_Avg'] > 0].sort_values('Abweichung_Avg', ascending=False)
kuerzer = merged[merged['Abweichung_Avg'] < 0].sort_values('Abweichung_Avg')

print(" IST länger als SOLL (Top 5):")
for _, r in laenger.head(5).iterrows():
    print(f"  {r['ORT']:<20} ({int(r['PLZ'])})  +{r['Abweichung_Avg']:.1f} Min")

print("\n IST kürzer als SOLL (Top 5):")
for _, r in kuerzer.head(5).iterrows():
    print(f"  {r['ORT']:<20} ({int(r['PLZ'])})  {r['Abweichung_Avg']:.1f} Min")

print(f" Mittlere Abweichung gesamt:  {merged['Abweichung_Avg'].mean():.1f} Min")


 IST länger als SOLL (Top 5):
  Visselhoevede        (27374)  +22.0 Min
  Rastede              (26180)  +16.8 Min
  Hamburg              (22113)  +9.4 Min
  Brunsbüttel          (25541)  +7.1 Min
  Bad Segeberg         (23795)  +5.2 Min

 IST kürzer als SOLL (Top 5):
  Bad Oldesloe         (23843)  -17.0 Min
  Rellingen            (25462)  -13.8 Min
  Reinbek              (21465)  -13.3 Min
  Hamburg              (22453)  -11.8 Min
  Hamburg              (22525)  -10.7 Min
 Mittlere Abweichung gesamt:  -3.6 Min


In [41]:
# IST-Fahrerdaten laden
ist_fahrer = pd.read_csv('Ist_Fahrer.csv')
ist_fahrer = ist_fahrer.rename(columns={
    'Avg': 'Ist_Avg',
    'Median': 'Ist_Median',
    'Stopps': 'Ist_Stopps',
    'Max': 'Ist_Max'
})

# SOLL je Fahrer aggregieren
soll_fahrer = (
    soll_raw[soll_raw['STOPP_MIN'] > 0]
    .groupby('FAHRERNAME')['STOPP_MIN']
    .agg(Soll_Avg='mean', Soll_Stopps='count')
    .reset_index()
)

# Mergen über Fahrername
merged_fahrer = pd.merge(
    soll_fahrer,
    ist_fahrer,
    left_on='FAHRERNAME',
    right_on='Fahrername',
    how='inner'
)

merged_fahrer['Abweichung'] = merged_fahrer['Ist_Avg'] - merged_fahrer['Soll_Avg']

merged_fahrer['Vollname'] = merged_fahrer['FAHRERNAME'].str.strip()

merged_fahrer = merged_fahrer.sort_values('Ist_Avg', ascending=True)

# Chart
fig = go.Figure()

fig.add_trace(go.Bar(
    y=merged_fahrer['Vollname'],
    x=merged_fahrer['Soll_Avg'].round(1),
    name='SOLL Ø',
    orientation='h',
    marker_color='#ff0000',
    opacity=0.85
))

fig.add_trace(go.Bar(
    y=merged_fahrer['Vollname'],
    x=merged_fahrer['Ist_Avg'].round(1),
    name='IST Ø',
    orientation='h',
    marker_color='#00498b',
    opacity=0.75,
    text=merged_fahrer['Abweichung'].round(1).apply(lambda v: f'{v:+.1f} Min'),
    textposition='outside',
    textfont=dict(color='#00498b', size=11)
))

fig.update_layout(
    template='plotly_white',
    barmode='group',
    title='Ø Standzeit pro Fahrer: SOLL vs. IST',
    xaxis_title='Ø Stoppdauer (Minuten)',
    yaxis_title='Fahrer',
    height=max(400, len(merged_fahrer) * 50),
    margin=dict(r=110),
    xaxis=dict(range=[0, merged_fahrer['Ist_Avg'].max() * 1.3]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()